⚡ Intermédiaire | ⏱ 2 h | 🔑 `class Fille(Mere)`, `super()`, redéfinition, polymorphisme, duck typing, ABC, composition

# 05 - Héritage, polymorphisme et composition

À la fin : tu sais créer une classe à partir d'une autre, redéfinir ou étendre son comportement, écrire du code qui traite des objets différents de la même façon, et choisir entre hériter et composer.


## 1. Hériter : `class Fille(Mere)`

L'héritage crée une classe **spécialisée** à partir d'une classe générale : la fille récupère tout (attributs, méthodes) et peut ajouter ou modifier. Test de bon sens : si on peut dire « un Chat **est un** Animal », l'héritage convient.

Vocabulaire : classe mère, parente, de base ; classe fille, enfant, dérivée, sous-classe.


In [ ]:
# La classe fille hérite de tout, sans rien écrire
class Animal:
    def __init__(self, nom):
        self.nom = nom

    def manger(self):
        return f"{self.nom} mange"

class Chat(Animal):
    pass

felix = Chat("Félix")
print(felix.manger())
print(isinstance(felix, Chat), isinstance(felix, Animal), issubclass(Chat, Animal))


In [ ]:
# Ajouter du comportement propre à la fille
class Chat(Animal):
    def ronronner(self):
        return f"{self.nom} ronronne"

felix = Chat("Félix")
print(felix.manger(), "|", felix.ronronner())


## 2. `super()` : réutiliser le constructeur parent

Quand la fille a ses propres attributs, elle redéfinit `__init__`. Elle doit d'abord laisser la mère initialiser les siens : `super().__init__(...)`.


In [ ]:
# super().__init__ : la mère initialise sa part, la fille ajoute la sienne
class Chien(Animal):
    def __init__(self, nom, race):
        super().__init__(nom)          # nom
        self.race = race               # attribut propre

rex = Chien("Rex", "berger")
print(rex.nom, rex.race, rex.manger())


In [ ]:
# L'héritage se chaîne : chaque niveau ajoute sa spécialité
class Vehicule:
    def __init__(self, marque):
        self.marque = marque

class Voiture(Vehicule):
    def __init__(self, marque, portes):
        super().__init__(marque)
        self.portes = portes

class VoitureElectrique(Voiture):
    def __init__(self, marque, portes, autonomie):
        super().__init__(marque, portes)
        self.autonomie = autonomie

ve = VoitureElectrique("Tesla", 5, 500)
print(vars(ve))
print([c.__name__ for c in VoitureElectrique.__mro__])   # ordre de recherche des attributs


## 3. Redéfinir une méthode

La fille peut **remplacer** une méthode héritée (même nom, même paramètres) ou l'**étendre** : appeler la version de la mère avec `super().methode()` puis compléter.


In [ ]:
# Remplacer : la version de la fille est utilisée à la place de celle de la mère
class Animal:
    def __init__(self, nom):
        self.nom = nom

    def crier(self):
        return "..."

class Chien(Animal):
    def crier(self):
        return "Wouf"

print(Animal("truc").crier(), Chien("Rex").crier())


In [ ]:
# Étendre : garder le comportement du parent et compléter
class Employe:
    def __init__(self, nom, salaire):
        self.nom = nom
        self.salaire = salaire

    def paie(self):
        return self.salaire

class Manager(Employe):
    def __init__(self, nom, salaire, bonus):
        super().__init__(nom, salaire)
        self.bonus = bonus

    def paie(self):
        return super().paie() + self.bonus        # réutilise puis ajoute

print(Employe("Ana", 2000).paie(), Manager("Bo", 3000, 500).paie())


## 4. Polymorphisme : même appel, comportements différents

Une liste d'objets de classes différentes, un seul appel `obj.crier()` : chaque objet répond à sa façon. Le code appelant n'a pas besoin de tester le type. Ajouter une nouvelle classe ne demande pas de modifier la boucle.


In [ ]:
# Le même message, une réponse par classe
class Chat(Animal):
    def crier(self):
        return "Miaou"

class Vache(Animal):
    def crier(self):
        return "Meuh"

for animal in [Chien("Rex"), Chat("Félix"), Vache("Marguerite")]:
    print(animal.nom, ":", animal.crier())


In [ ]:
# Une fonction écrite une fois pour toutes les classes, présentes et futures
def total_paie(employes):
    return sum(e.paie() for e in employes)

print(total_paie([Employe("Ana", 2000), Manager("Bo", 3000, 500)]))


In [ ]:
# ❌ Sans polymorphisme : une chaîne de if à compléter à chaque nouvelle classe
def crier(animal):
    if isinstance(animal, Chien):
        return "Wouf"
    elif isinstance(animal, Chat):
        return "Miaou"
    return "..."

# ✅ Avec polymorphisme : chaque classe porte son comportement
print(crier(Vache("Marguerite")), Vache("Marguerite").crier())


## 5. Duck typing

En Python, le polymorphisme ne demande même pas d'héritage : si l'objet **a la méthode**, ça marche (« si ça cancane comme un canard, c'est un canard »). Les fonctions natives font pareil : `len()` accepte tout objet qui a `__len__`.


In [ ]:
# Pas de parent commun : seule la présence de la méthode compte
class Canard:
    def crier(self):
        return "Coin"

class Robot:
    def crier(self):
        return "Bip"

for truc in [Canard(), Robot(), Chien("Rex")]:
    print(truc.crier())


## 6. Classes abstraites : imposer un contrat

Quand on veut **obliger** les filles à implémenter une méthode, on déclare la mère abstraite avec `ABC` et `@abstractmethod`. Elle ne peut pas être instanciée ; une fille qui oublie la méthode non plus. C'est l'équivalent Python d'une **interface** : « toute Forme sait calculer son aire ».


In [ ]:
# ABC + @abstractmethod : le contrat, plus une méthode concrète partagée
from abc import ABC, abstractmethod
import math

class Forme(ABC):
    @abstractmethod
    def aire(self):
        ...

    def decrire(self):                         # concrète : utilise le contrat
        return f"{type(self).__name__} d'aire {self.aire():.1f}"

class Carre(Forme):
    def __init__(self, cote):
        self.cote = cote

    def aire(self):
        return self.cote ** 2

class Disque(Forme):
    def __init__(self, rayon):
        self.rayon = rayon

    def aire(self):
        return math.pi * self.rayon ** 2

for f in [Carre(2), Disque(1)]:
    print(f.decrire())


In [ ]:
# Impossible d'instancier le contrat, ni une fille incomplète
class Triangle(Forme):
    pass                                       # aire() manquante

for cls in (Forme, Triangle):
    try:
        cls()
    except TypeError as e:
        print("TypeError :", e)


## 7. Composition : « a un » plutôt que « est un »

Si la phrase naturelle est « une Voiture **a un** Moteur », on ne fait pas hériter : on range un objet **dans** un autre et on lui **délègue** le travail. La composition est plus souple que l'héritage : on peut changer le composant, même pendant l'exécution.

Règle : **préférer la composition** ; garder l'héritage pour les vraies relations « est un ».


In [ ]:
# ❌ Héritage abusif : une voiture n'est pas un moteur
class Moteur:
    def demarrer(self):
        return "vroum"

class Voiture(Moteur):
    pass

# ✅ Composition : la voiture possède un moteur et lui délègue
class Voiture:
    def __init__(self):
        self.moteur = Moteur()

    def demarrer(self):
        return self.moteur.demarrer()

print(Voiture().demarrer())


In [ ]:
# La composition rend le comportement interchangeable
class Email:
    def envoyer(self, msg):
        return f"📧 {msg}"

class SMS:
    def envoyer(self, msg):
        return f"📱 {msg}"

class Utilisateur:
    def __init__(self, nom, canal):
        self.nom = nom
        self.canal = canal                  # composant reçu de l'extérieur

    def notifier(self, msg):
        return self.canal.envoyer(f"{self.nom} : {msg}")

u = Utilisateur("Ana", Email())
print(u.notifier("bienvenue"))
u.canal = SMS()                             # changement à l'exécution
print(u.notifier("code 1234"))


Deux nuances utiles pour les diagrammes UML du notebook 09 :

- **Composition** (losange plein ◆) : le composant est créé et détruit avec le tout. `Voiture` crée son `Moteur` dans `__init__`.
- **Agrégation** (losange vide ◇) : le tout regroupe des objets qui existent sans lui. Une `Equipe` reçoit des `Joueur` déjà créés.


In [ ]:
# Agrégation : les joueurs préexistent et survivent à l'équipe
class Joueur:
    def __init__(self, nom):
        self.nom = nom

class Equipe:
    def __init__(self, nom, joueurs):
        self.nom = nom
        self.joueurs = list(joueurs)        # reçus de l'extérieur

ana, bo = Joueur("Ana"), Joueur("Bo")
equipe = Equipe("A", [ana, bo])
del equipe                                  # les joueurs existent toujours
print(ana.nom, bo.nom)


## 8. Héritage multiple (aperçu)

Python autorise `class C(A, B)`. L'ordre de recherche des méthodes (*MRO*) va de gauche à droite, visible avec `C.__mro__`. Utile pour les *mixins* (petites classes qui ajoutent une capacité) ; à éviter pour tout le reste. Les détails sont hors programme.


In [ ]:
# Héritage multiple : le canard vole ET nage ; en cas de conflit, la première classe gagne
class Volant:
    def se_deplacer(self):
        return "vole"

class Nageur:
    def se_deplacer(self):
        return "nage"

class Canard(Volant, Nageur):
    pass

print(Canard().se_deplacer(), [c.__name__ for c in Canard.__mro__])


## ⚠️ Pièges

- Oublier `super().__init__()` : les attributs de la mère n'existent pas, `AttributeError` plus loin.
- Hériter pour « récupérer du code » alors que la relation est « a un » : composer.
- Hiérarchies profondes (5 niveaux) : illisible et fragile. Deux ou trois niveaux maximum.
- Hériter d'un type natif (`class MaListe(list)`) pour modifier une méthode : les autres méthodes (`extend`, `+=`) contournent ta redéfinition. Composer.
- Une fille qui **casse** le contrat de la mère : le code écrit pour la mère ne marche plus avec la fille (principe de substitution de Liskov). Exemple classique : `Carre(Rectangle)`.
- Redéfinir une méthode avec des **paramètres différents** : les appels polymorphes cassent.


In [ ]:
# BUG classique : super().__init__() oublié
class Parent:
    def __init__(self):
        self.valeur = 42

class Enfant(Parent):
    def __init__(self):
        self.autre = 1                    # ❌ Parent.__init__ jamais appelé

try:
    print(Enfant().valeur)
except AttributeError as e:
    print("AttributeError :", e)


In [ ]:
# BUG classique : hériter de list ne couvre pas tous les chemins d'ajout
class ListeLog(list):
    def append(self, x):
        print("ajout", x)
        super().append(x)

l = ListeLog()
l.append(1)              # loggé
l.extend([2, 3])         # pas loggé : extend n'appelle pas append
print(l)


In [ ]:
# BUG classique : Carre(Rectangle) casse une fonction écrite pour Rectangle
class Rectangle:
    def __init__(self, l, h):
        self.l, self.h = l, h

    def redimensionner(self, l, h):
        self.l, self.h = l, h

    def aire(self):
        return self.l * self.h

class Carre(Rectangle):
    def redimensionner(self, l, h):
        self.l = self.h = l               # force l == h

def doubler_largeur(r):                   # écrit en pensant à Rectangle
    r.redimensionner(r.l * 2, r.h)
    return r.aire()

print(doubler_largeur(Rectangle(2, 3)), doubler_largeur(Carre(2, 2)))   # 12 et 8 attendus, 16 obtenu
